This function contains the edge detection logic used in auto_estimate_chirp. It can be used to optimize the parameters for your system.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.ndimage import uniform_filter1d

def debug_trace_analysis(dataset, i=0, edge_width_fs=500):
    A = dataset.data
    t = A.time.to_numpy()
    # Extract the trace for the requested wavelength index
    trace = A.to_numpy()[:, i]
    wv_val = A.spectral.to_numpy()[i]

    # 1. Smoothing
    smoothed = savgol_filter(trace, 11, 2)
    
    # 2. Gradient (Raw Slope)
    grad = np.gradient(smoothed)
    
    # 3. Sustained Gradient (Moving Average)
    # Automatically calculate dt based on your time array (assumes t is in ps)
    dt = np.median(np.diff(t)) 
    window_size = max(1, int((edge_width_fs / 1000.0) / dt))
    sustained_grad = uniform_filter1d(grad, size=window_size)
    
    # 4. Find the peak of the edge (steepest part)
    idx_max = np.argmax(np.abs(sustained_grad))
    t0_found = t[idx_max]

    # 5. NEW: Find the start of the edge by backtracking
    # We walk backwards from idx_max until the raw gradient drops below 10% 
    # of its peak value, or changes sign (crosses zero).
    peak_grad_mag = np.abs(grad[idx_max])
    threshold_grad = 0.10 * peak_grad_mag  # 10% threshold
    
    idx_start = idx_max
    while idx_start > 0 and np.abs(sustained_grad[idx_start]) > threshold_grad:
        # Stop early if the slope changes direction (we hit the pre-pulse baseline)
        if grad[idx_start] * grad[idx_max] < 0:
            break
        idx_start -= 1
        
    t_start = t[idx_start]

    # --- Plotting ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    # Top Plot: Data
    ax1.plot(t, trace, label='Raw Data', alpha=0.3, color='gray')
    ax1.plot(t, smoothed, label='SavGol Smoothed', color='blue', lw=2)
    
    # Plot both the start and the peak
    ax1.axvline(t_start, color='orange', linestyle='--', label=f'Edge Start: {t_start:.3f} ps')
    ax1.axvline(t0_found, color='red', linestyle='--', label=f'Edge Peak: {t0_found:.3f} ps')
    
    ax1.set_ylabel('$\Delta$A Signal')
    ax1.set_title(f'Trace Analysis for $\lambda$ = {wv_val:.1f} nm (Index {i})')
    ax1.legend()

    # Bottom Plot: Derivatives
    ax2.plot(t, grad, label='Raw Gradient', alpha=0.4, color='green')
    ax2.plot(t, sustained_grad, label=f'Sustained Gradient ({edge_width_fs}fs)', color='darkgreen', lw=2)
    
    ax2.axvline(t_start, color='orange', linestyle='--')
    ax2.axvline(t0_found, color='red', linestyle='--')
    
    # Add a horizontal line to show the threshold we used to find the start
    if grad[idx_max] > 0:
        ax2.axhline(threshold_grad, color='orange', linestyle=':', alpha=0.5, label='10% Threshold')
    else:
        ax2.axhline(-threshold_grad, color='orange', linestyle=':', alpha=0.5, label='10% Threshold')

    ax2.set_ylabel('Gradient Magnitude')
    ax2.set_xlabel('Time (ps)')
    ax2.legend()

    plt.tight_layout()
    plt.xlim(-1, 10)
    plt.show()

# Run it for the first wavelength
debug_trace_analysis(dataset, i=0)